# Spatial Cross-Validation for Geographic Medical Data## Introduction to Spatial Autocorrelation in Medical DataWhen medical data has geographic or anatomical spatial structure, nearby observations are often more similar than distant ones (spatial autocorrelation). Standard cross-validation can be optimistically biased because:- **Geographic clustering**: Disease outbreaks, environmental factors- **Healthcare systems**: Regional treatment protocols- **Population genetics**: Ancestry patterns by geography- **Environmental exposure**: Air quality, water contaminationThis notebook demonstrates proper spatial cross-validation techniques for medical applications.

In [ ]:
# Import required librariesimport numpy as npimport pandas as pdimport matplotlib.pyplot as pltimport seaborn as snsfrom sklearn.datasets import make_blobsfrom sklearn.ensemble import RandomForestClassifierfrom sklearn.metrics import accuracy_score, roc_auc_scorefrom sklearn.model_selection import KFoldimport warningswarnings.filterwarnings('ignore')# Import our spatial CV methodsimport syssys.path.append('..')from trustcv.splitters.spatial import (    SpatialBlockCV, BufferedSpatialCV,     SpatiotemporalBlockCV, EnvironmentalHealthCV)# Set style and random seedplt.style.use('seaborn-v0_8-darkgrid')sns.set_palette(['#870052', '#FF876F', '#4CAF50', '#2196F3'])np.random.seed(42)

## 1. Creating Spatially Structured Medical DataLet's create synthetic data that mimics real-world medical scenarios with spatial patterns.

In [ ]:
def create_disease_outbreak_data(n_samples=1000, n_centers=5):    """    Create synthetic disease outbreak data with spatial clustering    """    # Generate clustered geographic coordinates    coords, cluster_labels = make_blobs(        n_samples=n_samples,         centers=n_centers,        n_features=2,  # latitude, longitude        cluster_std=0.8,        random_state=42    )        # Scale to realistic coordinates (Stockholm region)    lat_base, lon_base = 59.3293, 18.0686  # Stockholm coordinates    lat_range, lon_range = 0.5, 0.8  # Degree range        latitudes = lat_base + (coords[:, 0] - coords[:, 0].mean()) * lat_range / coords[:, 0].std()    longitudes = lon_base + (coords[:, 1] - coords[:, 1].mean()) * lon_range / coords[:, 1].std()        # Create features influenced by geography    features = []        # Environmental factors (spatially correlated)    air_quality_base = np.random.randn(n_centers)    air_quality = air_quality_base[cluster_labels] + np.random.randn(n_samples) * 0.3        water_quality_base = np.random.randn(n_centers)    water_quality = water_quality_base[cluster_labels] + np.random.randn(n_samples) * 0.3        # Socioeconomic factors (regional patterns)    income_base = np.random.randn(n_centers)    income_level = income_base[cluster_labels] + np.random.randn(n_samples) * 0.4        # Individual factors (less spatially correlated)    age = np.random.normal(50, 15, n_samples)    age = np.clip(age, 18, 85)        sex = np.random.binomial(1, 0.5, n_samples)        # Create target variable influenced by spatial factors    disease_prob = (        0.1 +  # Base probability        0.3 * (air_quality < -0.5) +  # Poor air quality increases risk        0.2 * (water_quality < -0.5) +  # Poor water quality increases risk        0.15 * (income_level < -0.5) +  # Low income increases risk        0.1 * (age > 65) +  # Elderly at higher risk        np.random.randn(n_samples) * 0.1  # Random noise    )        # Convert to binary outcome    disease = (disease_prob > np.random.random(n_samples)).astype(int)        # Create DataFrame    df = pd.DataFrame({        'latitude': latitudes,        'longitude': longitudes,        'cluster': cluster_labels,        'air_quality': air_quality,        'water_quality': water_quality,        'income_level': income_level,        'age': age,        'sex': sex,        'disease': disease    })        return df# Generate the datasetdf = create_disease_outbreak_data(n_samples=1000, n_centers=6)print(f"Dataset shape: {df.shape}")print(f"Disease prevalence: {df['disease'].mean():.2%}")print(f"Geographic range: Lat {df['latitude'].min():.3f} to {df['latitude'].max():.3f}")print(f"Geographic range: Lon {df['longitude'].min():.3f} to {df['longitude'].max():.3f}")

In [ ]:
# Visualize the spatial distributionfig, axes = plt.subplots(1, 3, figsize=(18, 5))# Plot 1: Geographic distribution of casesax1 = axes[0]scatter = ax1.scatter(df['longitude'], df['latitude'],                      c=df['disease'], cmap='RdYlBu_r',                      alpha=0.7, s=50)ax1.set_xlabel('Longitude')ax1.set_ylabel('Latitude')ax1.set_title('Disease Distribution by Location')plt.colorbar(scatter, ax=ax1, label='Disease Status')# Plot 2: Disease rate by clusterax2 = axes[1]cluster_rates = df.groupby('cluster')['disease'].mean().sort_values()bars = ax2.bar(range(len(cluster_rates)), cluster_rates.values,                color=['#4CAF50' if r < 0.3 else '#FF876F' if r < 0.5 else '#870052'                      for r in cluster_rates.values])ax2.set_xlabel('Geographic Cluster')ax2.set_ylabel('Disease Rate')ax2.set_title('Disease Rate by Geographic Cluster')ax2.set_xticks(range(len(cluster_rates)))ax2.set_xticklabels([f'Cluster {i}' for i in cluster_rates.index])# Add value labelsfor i, bar in enumerate(bars):    height = bar.get_height()    ax2.text(bar.get_x() + bar.get_width()/2., height + 0.01,             f'{height:.2%}', ha='center', va='bottom')# Plot 3: Correlation matrix of spatial factorsax3 = axes[2]spatial_factors = ['air_quality', 'water_quality', 'income_level', 'disease']corr_matrix = df[spatial_factors].corr()im = ax3.imshow(corr_matrix, cmap='RdBu_r', aspect='equal', vmin=-1, vmax=1)ax3.set_xticks(range(len(spatial_factors)))ax3.set_yticks(range(len(spatial_factors)))ax3.set_xticklabels(spatial_factors, rotation=45)ax3.set_yticklabels(spatial_factors)ax3.set_title('Correlation Matrix')# Add correlation valuesfor i in range(len(spatial_factors)):    for j in range(len(spatial_factors)):        ax3.text(j, i, f'{corr_matrix.iloc[i, j]:.2f}',                 ha='center', va='center',                 color='white' if abs(corr_matrix.iloc[i, j]) > 0.5 else 'black')plt.colorbar(im, ax=ax3, label='Correlation')plt.tight_layout()plt.show()print("\n📊 Key Observations:")print(f"• Disease rates vary significantly by cluster: {cluster_rates.min():.1%} to {cluster_rates.max():.1%}")print(f"• Spatial autocorrelation detected in environmental factors")print(f"• Standard CV may be overly optimistic due to spatial clustering")

## 2. The Problem with Standard Cross-ValidationLet's demonstrate why standard cross-validation fails with spatially structured data.

In [ ]:
# Prepare features and targetfeature_cols = ['air_quality', 'water_quality', 'income_level', 'age', 'sex']X = df[feature_cols].valuesy = df['disease'].valuescoordinates = df[['longitude', 'latitude']].valuesprint("Comparing Standard vs Spatial Cross-Validation")print("=" * 50)# 1. Standard K-Fold (WRONG for spatial data)print("\n1. Standard K-Fold (Ignores Spatial Structure)")print("-" * 40)kfold = KFoldMedical(n_splits=5, shuffle=True, random_state=42)standard_scores = []for fold, (train_idx, test_idx) in enumerate(kfold.split(X), 1):    # Train model    model = RandomForestClassifier(n_estimators=100, random_state=42)    model.fit(X[train_idx], y[train_idx])        # Evaluate    y_pred_proba = model.predict_proba(X[test_idx])[:, 1]    score = roc_auc_score(y[test_idx], y_pred_proba)    standard_scores.append(score)        # Check spatial mixing    train_coords = coordinates[train_idx]    test_coords = coordinates[test_idx]        print(f"Fold {fold}: ROC-AUC = {score:.4f}")print(f"\nMean ROC-AUC: {np.mean(standard_scores):.4f} ± {np.std(standard_scores):.4f}")print("⚠️  This is likely OPTIMISTICALLY BIASED due to spatial autocorrelation!")

## 3. Spatial Block Cross-ValidationThe first spatial CV method creates geographic blocks to reduce spatial autocorrelation between train and test sets.

In [ ]:
print("\n2. Spatial Block Cross-Validation")print("-" * 40)# Spatial Block CVspatial_cv = SpatialBlockCV(n_splits=5, block_size=0.1)  # 0.1 degree blocksspatial_scores = []for fold, (train_idx, test_idx) in enumerate(spatial_cv.split(X, coordinates=coordinates), 1):    # Train model    model = RandomForestClassifier(n_estimators=100, random_state=42)    model.fit(X[train_idx], y[train_idx])        # Evaluate    y_pred_proba = model.predict_proba(X[test_idx])[:, 1]    score = roc_auc_score(y[test_idx], y_pred_proba)    spatial_scores.append(score)        print(f"Fold {fold}: ROC-AUC = {score:.4f} (train: {len(train_idx)}, test: {len(test_idx)})")print(f"\nMean ROC-AUC: {np.mean(spatial_scores):.4f} ± {np.std(spatial_scores):.4f}")print("✅ More realistic estimate accounting for spatial structure")# Compare the biasbias = np.mean(standard_scores) - np.mean(spatial_scores)print(f"\n📊 Spatial Bias Estimate: {bias:.4f}")if bias > 0.05:    print("⚠️  Significant optimistic bias in standard CV!")    print("   → Always use spatial CV for geographic data")else:    print("✅ Minimal spatial bias detected")

In [ ]:
# Visualize spatial blocks for one foldfig, axes = plt.subplots(1, 2, figsize=(15, 6))# Get one fold for visualizationspatial_cv_viz = SpatialBlockCV(n_splits=5, block_size=0.1)train_idx, test_idx = next(spatial_cv_viz.split(X, coordinates=coordinates))# Plot 1: Standard CV fold (random mixing)ax1 = axes[0]kfold_viz = KFoldMedical(n_splits=5, shuffle=True, random_state=42)train_std, test_std = next(kfold_viz.split(X))ax1.scatter(coordinates[train_std, 0], coordinates[train_std, 1],            c='#4CAF50', alpha=0.6, s=30, label='Train')ax1.scatter(coordinates[test_std, 0], coordinates[test_std, 1],            c='#FF876F', alpha=0.8, s=30, label='Test')ax1.set_xlabel('Longitude')ax1.set_ylabel('Latitude')ax1.set_title('Standard K-Fold: Random Spatial Mixing')ax1.legend()ax1.grid(True, alpha=0.3)# Plot 2: Spatial CV fold (blocked structure)ax2 = axes[1]ax2.scatter(coordinates[train_idx, 0], coordinates[train_idx, 1],            c='#4CAF50', alpha=0.6, s=30, label='Train')ax2.scatter(coordinates[test_idx, 0], coordinates[test_idx, 1],            c='#FF876F', alpha=0.8, s=30, label='Test')ax2.set_xlabel('Longitude')ax2.set_ylabel('Latitude')ax2.set_title('Spatial Block CV: Geographic Separation')ax2.legend()ax2.grid(True, alpha=0.3)plt.tight_layout()plt.show()print("\n🔍 Key Differences:")print("• Standard CV: Train and test points are randomly mixed geographically")print("• Spatial Block CV: Train and test points are geographically separated")print("• Spatial separation provides more realistic performance estimates")

## 4. Buffered Spatial Cross-ValidationTo further reduce spatial autocorrelation, we can add buffer zones around test blocks.

In [ ]:
print("\n3. Buffered Spatial Cross-Validation")print("-" * 40)# Buffered Spatial CVbuffered_cv = BufferedSpatialCV(n_splits=5, buffer_size=0.05)  # 0.05 degree bufferbuffered_scores = []for fold, (train_idx, test_idx) in enumerate(buffered_cv.split(X, coordinates=coordinates), 1):    if len(test_idx) < 10:  # Skip folds with too few test samples        print(f"Fold {fold}: Skipped (insufficient test samples)")        continue            # Train model    model = RandomForestClassifier(n_estimators=100, random_state=42)    model.fit(X[train_idx], y[train_idx])        # Evaluate    y_pred_proba = model.predict_proba(X[test_idx])[:, 1]    score = roc_auc_score(y[test_idx], y_pred_proba)    buffered_scores.append(score)        print(f"Fold {fold}: ROC-AUC = {score:.4f} (train: {len(train_idx)}, test: {len(test_idx)})")if buffered_scores:    print(f"\nMean ROC-AUC: {np.mean(buffered_scores):.4f} ± {np.std(buffered_scores):.4f}")    print("✅ Most conservative estimate with buffer zones")else:    print("⚠️  Buffer too large for dataset - consider smaller buffer size")

## 5. Spatiotemporal Cross-ValidationFor medical surveillance data that has both spatial and temporal dimensions.

In [ ]:
# Create spatiotemporal data (extending our dataset)def create_spatiotemporal_data(df, n_time_points=12):    """    Extend spatial data with temporal dimension (monthly surveillance)    """    spatiotemporal_data = []        for month in range(n_time_points):        monthly_df = df.copy()        monthly_df['month'] = month        monthly_df['time_index'] = month                # Add temporal trends        seasonal_effect = 0.1 * np.sin(2 * np.pi * month / 12)  # Seasonal pattern        trend_effect = 0.05 * month / n_time_points  # Linear trend                # Modify disease probability with temporal effects        base_prob = monthly_df['disease']        temporal_prob = np.clip(            base_prob + seasonal_effect + trend_effect + np.random.randn(len(df)) * 0.05,            0, 1        )        monthly_df['disease_temporal'] = (temporal_prob > 0.3).astype(int)                spatiotemporal_data.append(monthly_df)        return pd.concat(spatiotemporal_data, ignore_index=True)# Create spatiotemporal datasetst_df = create_spatiotemporal_data(df, n_time_points=12)print(f"Spatiotemporal dataset shape: {st_df.shape}")print(f"Time points: {st_df['month'].nunique()}")print(f"Disease prevalence over time: {st_df['disease_temporal'].mean():.2%}")

In [ ]:
print("\n4. Spatiotemporal Block Cross-Validation")print("-" * 40)# Prepare spatiotemporal featuresst_feature_cols = ['air_quality', 'water_quality', 'income_level', 'age', 'sex', 'month']X_st = st_df[st_feature_cols].valuesy_st = st_df['disease_temporal'].valuescoordinates_st = st_df[['longitude', 'latitude']].valuestimes_st = st_df['time_index'].values# Spatiotemporal CVst_cv = SpatiotemporalBlockCV(n_splits=5, spatial_block_size=0.08, temporal_block_size=3)st_scores = []for fold, (train_idx, test_idx) in enumerate(st_cv.split(X_st, coordinates=coordinates_st, times=times_st), 1):    if len(test_idx) < 20:        print(f"Fold {fold}: Skipped (insufficient test samples)")        continue            # Train model    model = RandomForestClassifier(n_estimators=50, random_state=42)    model.fit(X_st[train_idx], y_st[train_idx])        # Evaluate    y_pred_proba = model.predict_proba(X_st[test_idx])[:, 1]    score = roc_auc_score(y_st[test_idx], y_pred_proba)    st_scores.append(score)        # Show temporal and spatial coverage    train_months = set(times_st[train_idx])    test_months = set(times_st[test_idx])        print(f"Fold {fold}: ROC-AUC = {score:.4f}")    print(f"  Train months: {sorted(train_months)}")    print(f"  Test months: {sorted(test_months)}")if st_scores:    print(f"\nMean ROC-AUC: {np.mean(st_scores):.4f} ± {np.std(st_scores):.4f}")    print("✅ Accounts for both spatial and temporal autocorrelation")

## 6. Environmental Health Cross-ValidationSpecialized method for environmental health studies with exposure gradients.

In [ ]:
print("\n5. Environmental Health Cross-Validation")print("-" * 40)# Environmental Health CV (considers exposure zones)env_cv = EnvironmentalHealthCV(n_splits=5, exposure_buffer=0.08)env_scores = []# Create exposure source (e.g., industrial site)exposure_source = np.array([df['longitude'].mean(), df['latitude'].mean()])for fold, (train_idx, test_idx) in enumerate(env_cv.split(X, coordinates=coordinates, exposure_source=exposure_source), 1):    if len(test_idx) < 10:        print(f"Fold {fold}: Skipped (insufficient test samples)")        continue            # Train model    model = RandomForestClassifier(n_estimators=100, random_state=42)    model.fit(X[train_idx], y[train_idx])        # Evaluate    y_pred_proba = model.predict_proba(X[test_idx])[:, 1]    score = roc_auc_score(y[test_idx], y_pred_proba)    env_scores.append(score)        # Calculate distance to exposure source for each set    train_coords = coordinates[train_idx]    test_coords = coordinates[test_idx]        train_distances = np.linalg.norm(train_coords - exposure_source, axis=1)    test_distances = np.linalg.norm(test_coords - exposure_source, axis=1)        print(f"Fold {fold}: ROC-AUC = {score:.4f}")    print(f"  Train distance from source: {train_distances.mean():.3f} ± {train_distances.std():.3f}")    print(f"  Test distance from source: {test_distances.mean():.3f} ± {test_distances.std():.3f}")if env_scores:    print(f"\nMean ROC-AUC: {np.mean(env_scores):.4f} ± {np.std(env_scores):.4f}")    print("✅ Specialized for environmental exposure studies")

## 7. Comparison of All Spatial MethodsLet's compare all the spatial cross-validation methods we've implemented.

In [ ]:
# Compile all resultsresults = {    'Standard K-Fold (BIASED)': {'scores': standard_scores, 'color': '#FF4444'},    'Spatial Block CV': {'scores': spatial_scores, 'color': '#870052'},    'Buffered Spatial CV': {'scores': buffered_scores, 'color': '#FF876F'},    'Spatiotemporal CV': {'scores': st_scores, 'color': '#4CAF50'},    'Environmental Health CV': {'scores': env_scores, 'color': '#2196F3'}}# Filter out empty resultsresults = {k: v for k, v in results.items() if v['scores']}# Create visualizationfig, axes = plt.subplots(1, 2, figsize=(15, 6))# Plot 1: Box plot comparisonax1 = axes[0]data_for_box = []labels = []colors = []for method, data in results.items():    if data['scores']:  # Only include non-empty results        data_for_box.append(data['scores'])        labels.append(method)        colors.append(data['color'])if data_for_box:    bp = ax1.boxplot(data_for_box, labels=labels, patch_artist=True)    for patch, color in zip(bp['boxes'], colors):        patch.set_facecolor(color)        patch.set_alpha(0.7)ax1.set_ylabel('ROC-AUC Score')ax1.set_title('Spatial Cross-Validation Methods Comparison')ax1.grid(True, alpha=0.3)ax1.tick_params(axis='x', rotation=45)# Plot 2: Mean scores with error barsax2 = axes[1]methods = list(results.keys())means = [np.mean(results[m]['scores']) for m in methods]stds = [np.std(results[m]['scores']) for m in methods]colors_list = [results[m]['color'] for m in methods]x_pos = np.arange(len(methods))bars = ax2.bar(x_pos, means, yerr=stds, capsize=10,               color=colors_list, alpha=0.7,               edgecolor='black', linewidth=1)ax2.set_xlabel('CV Method')ax2.set_ylabel('Mean ROC-AUC Score')ax2.set_title('Mean Performance ± Standard Deviation')ax2.set_xticks(x_pos)ax2.set_xticklabels(methods, rotation=45, ha='right')ax2.grid(True, alpha=0.3, axis='y')# Add value labelsfor bar, mean, std in zip(bars, means, stds):    height = bar.get_height()    ax2.text(bar.get_x() + bar.get_width()/2., height + std + 0.01,             f'{mean:.3f}', ha='center', va='bottom')plt.tight_layout()plt.show()# Print summaryprint("\n" + "=" * 60)print("SPATIAL CROSS-VALIDATION COMPARISON SUMMARY")print("=" * 60)for method, data in results.items():    scores = data['scores']    print(f"{method:25s}: {np.mean(scores):.4f} ± {np.std(scores):.4f}")if 'Standard K-Fold (BIASED)' in results and len(results) > 1:    standard_mean = np.mean(results['Standard K-Fold (BIASED)']['scores'])    spatial_methods = [k for k in results.keys() if k != 'Standard K-Fold (BIASED)']        print("\n📊 Bias Analysis:")    for method in spatial_methods:        spatial_mean = np.mean(results[method]['scores'])        bias = standard_mean - spatial_mean        print(f"   {method}: {bias:+.4f} optimistic bias")print("\n💡 Key Takeaways:")print("   • Standard CV often overestimates performance for spatial data")print("   • Spatial block CV provides more realistic estimates")print("   • Buffer zones further reduce spatial autocorrelation")print("   • Choose method based on your specific spatial structure")

## 8. Best Practices for Spatial Cross-ValidationGuidelines for choosing and implementing spatial CV methods.

In [ ]:
print("SPATIAL CROSS-VALIDATION BEST PRACTICES")print("=" * 60)print("\n🎯 When to Use Spatial CV:")print("   ✅ Geographic medical data (disease surveillance, environmental health)")print("   ✅ Medical imaging with anatomical spatial structure")print("   ✅ Multi-site studies with regional effects")print("   ✅ Any data where distance correlates with similarity")print("\n📏 Choosing Block/Buffer Sizes:")print("   • Start with domain knowledge (disease transmission range)")print("   • Use variogram analysis to find spatial correlation range")print("   • Test multiple sizes and check stability")print("   • Balance between spatial separation and sample size")print("\n⚖️ Method Selection Guide:")print("   • Spatial Block CV: Standard choice for geographic data")print("   • Buffered Spatial CV: When strong spatial autocorrelation exists")print("   • Spatiotemporal CV: For longitudinal spatial data")print("   • Environmental Health CV: For exposure-based studies")print("\n⚠️ Common Pitfalls:")print("   ❌ Using standard CV for any spatial/geographic data")print("   ❌ Choosing blocks too small (insufficient spatial separation)")print("   ❌ Choosing blocks too large (insufficient sample size)")print("   ❌ Ignoring edge effects at study area boundaries")print("\n📊 Validation Checklist:")print("   ☑️ Test for spatial autocorrelation in residuals")print("   ☑️ Visualize train/test spatial distribution")print("   ☑️ Compare standard vs spatial CV performance")print("   ☑️ Report spatial bias magnitude")print("   ☑️ Document spatial parameters used")

## 9. Real-World Medical ApplicationsExamples of spatial CV applications in medical research.

In [ ]:
print("REAL-WORLD MEDICAL APPLICATIONS")print("=" * 50)applications = [    {        'title': '🦠 Disease Outbreak Prediction',        'description': 'Predict COVID-19 spread using mobility and demographic data',        'spatial_structure': 'Geographic clustering of cases',        'recommended_method': 'Buffered Spatial CV',        'key_considerations': ['Transmission radius', 'Population density', 'Mobility patterns']    },    {        'title': '🏭 Environmental Health Studies',        'description': 'Assess cancer risk near industrial facilities',        'spatial_structure': 'Distance-based exposure gradients',        'recommended_method': 'Environmental Health CV',        'key_considerations': ['Exposure radius', 'Wind patterns', 'Groundwater flow']    },    {        'title': '🧠 Neuroimaging Analysis',        'description': 'Classify brain tumors from MRI scans',        'spatial_structure': 'Anatomical spatial correlation',        'recommended_method': 'Spatial Block CV',        'key_considerations': ['Voxel neighborhoods', 'Anatomical regions', 'Spatial smoothing']    },    {        'title': '🏥 Healthcare Access Modeling',        'description': 'Predict healthcare utilization patterns',        'spatial_structure': 'Geographic healthcare catchments',        'recommended_method': 'Spatial Block CV',        'key_considerations': ['Hospital catchments', 'Transportation networks', 'Socioeconomic clusters']    },    {        'title': '🌡️ Climate-Health Relationships',        'description': 'Model heat-related illness risk',        'spatial_structure': 'Weather patterns and microclimates',        'recommended_method': 'Spatiotemporal Block CV',        'key_considerations': ['Climate zones', 'Urban heat islands', 'Seasonal patterns']    }]for i, app in enumerate(applications, 1):    print(f"\n{i}. {app['title']}")    print(f"   Description: {app['description']}")    print(f"   Spatial Structure: {app['spatial_structure']}")    print(f"   Recommended Method: {app['recommended_method']}")    print(f"   Key Considerations: {', '.join(app['key_considerations'])}")print("\n" + "=" * 50)print("Remember: The choice of spatial CV method should always be")print("guided by the underlying spatial process in your medical domain!")

## ConclusionSpatial cross-validation is essential for obtaining realistic performance estimates when working with geographic or spatially structured medical data. The key insights from this notebook are:1. **Standard CV is biased** for spatial data due to spatial autocorrelation2. **Spatial Block CV** provides more realistic estimates by creating geographic blocks3. **Buffered Spatial CV** adds safety margins around test areas4. **Spatiotemporal CV** handles both spatial and temporal dependencies5. **Environmental Health CV** is specialized for exposure-based studiesAlways consider the spatial structure in your medical data and choose the appropriate validation method accordingly. The goal is to estimate how well your model will perform when applied to new geographic regions or populations.### Next Steps:- Try the methods with your own spatial medical data- Experiment with different block sizes and buffer zones- Compare spatial CV results with standard CV to quantify bias- Consider hybrid approaches for complex spatial structures